# Interpreting the result gets from random forest classfier
* this notebook will be interpreting the results such as feature importances got from random forest.
* The goal is to be able to find the most important part of our data that contributes the most to the random forest classfier.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mne.decoding import Vectorizer
from sklearn.preprocessing import MinMaxScaler
from mne.decoding import SPoC
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split as tts
import mne
from sklearn.model_selection import StratifiedKFold
from mne.datasets import sample
from mne.decoding import (SlidingEstimator, GeneralizingEstimator,
                          cross_val_multiscore, LinearModel, get_coef)
from sklearn.decomposition import PCA
from sklearn.decomposition import NMF
from sklearn.manifold import TSNE 
import os
from mne.decoding import UnsupervisedSpatialFilter
import os.path as op
from os import listdir
from os.path import isfile, join
from collections import defaultdict
from scipy.stats import skew, kurtosis
import mne 

from mne.decoding import Vectorizer
from sklearn.preprocessing import MinMaxScaler, Normalizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.svm import SVC  # noqa
from sklearn.model_selection import ShuffleSplit  # noqa

from mne.decoding import UnsupervisedSpatialFilter
from sklearn.decomposition import PCA
from scipy.stats import skew, kurtosis
from sklearn.metrics import confusion_matrix
from mne.viz import tight_layout

import time
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
%matplotlib inline

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from urllib.request import urlopen

plt.style.use('ggplot')
pd.set_option('display.max_columns', 500)

In [2]:
%run Batch_ArtifactFilter_Epoch.ipynb

In [26]:
#Read Epoch
epoch=mne.read_epochs('D:\\USB Drive\\NewEEG-200s\\New\\20131216_1310_epo.fif')

Reading D:\USB Drive\NewEEG-200s\New\20131216_1310_epo.fif ...
    Found the data of interest:
        t =       0.00 ...     500.00 ms
        0 CTF compensation matrices available
1759 matching events found
Applying baseline correction (mode: mean)
1759 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated


In [5]:

## Methods for convert our events into binary, 4: Language, 5:Non Language
def filter_event_id(epoch):
    newepoch=epoch
    epochs=[[]]
    j=0
    for i in range(len(newepoch.events[:,-1])):
        if i % 3 ==0:
            epochs.append([])
            j=j+1
            epochs[j].append(epoch.events[i,-1])
        else:
            epochs[j].append(epoch.events[i,-1])
    for events in epochs:
        if  4 in events or 1 in events or 13 in events or 16 in events:
            for i in range(len(events)):
                    events[i]=1
        else:
            for i in range(len(events)):
                    events[i]=0
    epochs.pop(0)
    flattened_list = [y for x in epochs for y in x]
    for i in range(len(newepoch.events)):
        newepoch.events[i]=flattened_list[i]
    return newepoch

In [6]:
#Filter events 0: Nonlanguage 1: Non Language
epoch=filter_event_id(epoch)
print(epoch.events[:,-1])

[1 1 1 ... 1 1 1]


In [16]:
## Vecorize to convert 3d to 2d
X = epoch.get_data()
y = epoch.events[:,-1]
vec=Vectorizer()
X=vec.fit_transform(X)
print(X.shape)

(1759, 32125)


In [8]:

# Define a unique pipeline to sequentially:
clf = make_pipeline(Vectorizer(),
                   StandardScaler(),
                         LogisticRegression())
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state = 42)
    
clf.fit(X_train, y_train)
    
score = clf.score(X_test, y_test)

C:\Users\kingsly\Anaconda3\envs\mne\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


In [9]:
print(score)

0.5606060606060606


In [10]:
#Classcify the same data again using RandomForest
from sklearn import metrics
clf = RandomForestRegressor(n_estimators =10, random_state = 42)
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 42)
    
clf.fit(X_train, y_train)
    
y_pred=clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred.round()))

Accuracy: 0.9090909090909091


In [20]:
def removebadchannel(epoch):
    epoch.drop_channels(['Lm','Rm','VEOG','Nasium'])
    epoch.drop_channels(['STI 014'])

In [ ]:
#Remove bad Channels 
removebadchannel(epoch)

In [17]:
X = epoch.get_data()
y = epoch.events[:,-1]
vec=Vectorizer()
X=vec.fit_transform(X)
print(X.shape)

(1759, 32125)


In [ ]:
from sklearn import metrics
clf = RandomForestRegressor(n_estimators =10, random_state = 42)
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 42)
    
clf.fit(X_train, y_train)
    
y_pred=clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred.round()))
score = clf.score(X_test, y_test)
print(score)

In [22]:
y_pred

array([0.7, 0.7, 0.6, 0.2, 0.5, 0.8, 0.5, 0.3, 0.5, 0.5, 0.2, 0.5, 0.3,
       0.2, 0.7, 0.3, 0.6, 0.6, 0.3, 0.7, 0.7, 0.7, 0.6, 0.3, 0.4, 0.5,
       0.6, 0.3, 0.5, 0.5, 0.8, 0.6, 0.9, 0.4, 0.6, 0.6, 0. , 0.3, 0.3,
       0.3, 0.5, 0.3, 0.3, 0.4, 0.7, 0.4, 0.3, 0.6, 0.3, 0.6, 0.3, 0.7,
       0.4, 0.6, 0.8, 0.7, 0.3, 0.4, 0.5, 0.5, 0.5, 0.4, 0.1, 0.5, 0.7,
       0.6, 0.5, 0.6, 0.8, 0.4, 0.7, 0.8, 0.5, 0.4, 0.7, 0.4, 0.6, 0. ,
       0.5, 0.6, 0.8, 0.3, 0.6, 0.4, 0.6, 0.6, 0.5, 0.5, 0.4, 0.2, 0.6,
       0.8, 0.7, 0.4, 0.4, 0.5, 0.6, 0.5, 0.4, 0.3, 0.7, 0.4, 0.5, 0.3,
       0.1, 0.5, 0.7, 0.6, 0.4, 0.5, 0.2, 0.2, 0.3, 0.7, 0.3, 0.7, 0.4,
       0.2, 0.6, 0.3, 0.3, 0.2, 0.3, 0.3, 0.5, 0.3, 0.6, 0.4, 0.5, 0.9,
       0.3, 0.4, 0.5, 0.7, 0.4, 0.2, 0.5, 0.3, 0.5, 0.6, 0.3, 0.6, 0.6,
       0.8, 0.6, 0.3, 0.4, 0.5, 0.3, 0.5, 0.4, 0.2, 0.3, 0.6, 0.7, 0.6,
       0.3, 0.2, 0.5, 0.3, 0.4, 0.4, 0.3, 0.3, 0.6, 0.3, 0.3, 0.6, 0.5,
       0.4, 0.4, 0.4, 0.2, 0.4, 0.5, 0.2, 0.6, 0.5, 0.3, 0.5, 0.

In [23]:
epoch.drop_channels(['Nasium'])

<EpochsFIF  |   1759 events (all good), 0 - 0.5 sec, baseline [0, 0], ~428.0 MB, data loaded,
 'b-f1-pic': 0
 'b-f1-snd': 0
 'b-f1-spk': 0
 'b-f1-wrd': 0
 'b-f2-pic': 0
 'b-f2-snd': 0
 'b-f2-spk': 0
 'b-f2-wrd': 811
 'b-tg-pic': 0
 'b-tg-snd': 0
 'b-tg-spk': 0
 'b-tg-wrd': 0
 'f-f1-pic': 0
 'f-f1-snd': 0
 'f-f1-spk': 0
 'f-f1-wrd': 0
 'f-f2-pic': 0
 'f-f2-snd': 0
 'f-f2-spk': 0
 'f-f2-wrd': 0
 'f-tg-pic': 0
 'f-tg-snd': 0
 'f-tg-spk': 0
 'f-tg-wrd': 0>

In [32]:
def convert_event_ids_to_stim_combinations(epoch_object, trial_index_list, stim_combinations):
    """
        epoch_object: mne.epochs.EpochsFIF object
        ----
        trial_index_list: A list containing the start and end index for each trial of an epoch object
        ----
        stim_combinations: A dictionary whose keys are 3-tuples and values are corressponding stim-code combination value
                           ex. { (5,6,4) : "AALL" }
        ---
        returns
            List of Lists containing events for an epoch object. 
            
    """
    new_events_master = []
    copy_epoch = epoch_object.copy()
    print("events shape: {}".format(epoch_object.events.shape))
    print("")
    for i in range(len(trial_index_list)): # iterate through trial index list
    # extract start and end index

        start = trial_index_list[i][0]
        end = trial_index_list[i][1]

        #initialize temp epoch object between start - end indices
#         print(start,end)
        temp_epoch = copy_epoch[start:end]
        # get temp epoch events
        temp_epoch_events = temp_epoch.events
        temp_epoch_event_ids = temp_epoch_events[:,-1]
        
        stim_code_key = get_stim_code_combination(temp_epoch, stim_combinations)
        
        if stim_code_key != 'NaN':
            new_event_id = modality_lexicality_event_ids[stim_code_key]

            temp_epoch_merged_events = mne.merge_events(temp_epoch_events,
                                                        temp_epoch_event_ids,
                                                        new_event_id,
                                                        replace_events = True)

            new_events_master.append(temp_epoch_merged_events)

        else:
            
            print("Could not find stim-code combination.")
            print("error in trial index: {}".format(trial_index_list[i]))
            new_events_master.append(temp_epoch_events)
#             print("Meta: stim-combo-key: {0}".format(stim_code_key))
    
    
    new_events_master = np.array(new_events_master).reshape(-1,1)
    print(np.array(new_events_master).shape)
    return_array = np.concatenate([x for x in new_events_master])
    return np.concatenate([x for x in return_array])

In [ ]:
if 'Nasium' in epoch.ch_names:
    epoch.drop_channels(ch_names=['Nasium', 'LL4', 'L12', 'VEOG']);
copy_epoch = epoch.copy()
X = copy_epoch.pick_types(meg=False, eeg=True)
new_events = convert_epoch_events_to_stim_combinations(epoch_object= copy_epoch)    
copy_epoch.events = new_events

In [29]:
X.shape

AttributeError: 'EpochsFIF' object has no attribute 'shape'

In [34]:
def get_epoch_trials(epoch_object):
    #note: third stim-code has a value that is 1 more than the first.
    events = epoch_object.events
    found_first_stim_code = False
    found_second_stim_code = False
    found_third_stim_code = False
    current_trial = []
    trials = []
    for event in events:
        if not found_first_stim_code and not found_second_stim_code and not found_third_stim_code: #still searching for the end of trial
            current_trial.append(event)
            found_first_stim_code = True
            first_stim_code = event
        elif not found_second_stim_code and not found_third_stim_code:
            current_trial.append(event)
            found_second_stim_code = True
        elif not found_third_stim_code and (event[-1] + 1) == first_stim_code[-1]: 
            current_trial.append(event)
            found_third_stim_code = True
            trials.append(current_trial)
            current_trial = []
            found_first_stim_code = False
            found_second_stim_code = False
            found_third_stim_code = False
        elif not found_third_stim_code and (event[-1] + 1) != first_stim_code[-1]:
            # there is no third event, trial is unfinished.
            current_trial = []
            current_trial.append(event)#assuming it's first stim-code...for now
            found_first_stim_code = True
    return trials

In [35]:
def stim_code_combination_from_trial(stim_code_tuple, stim_combinations):
    """
    Description:
        returns the correct stim-code combination event id 
        example - a trial has events (20, 15, 19), this method will match those values to a 
        stim combination that is defined in the dictionary stim_combinations
    """
    for stim_code in stim_combinations.keys():
        if stim_code_tuple == stim_code:
            return stim_combinations[stim_code]
    return "NaN"

In [36]:
def convert_epoch_events_to_stim_combinations(epoch_object):
    trials = get_epoch_trials(epoch_object)
    new_events_master = []
    
    for trial in trials:
#         print(trial)
        trial = np.array(trial)
        stim_codes = trial[:,-1]
        try:
#             print(stim_codes)
            stim_code_tuple = (stim_codes[0],stim_codes[1], stim_codes[2])
#             print(stim_code_tuple)
            stim_combination = stim_code_combination_from_trial(stim_code_tuple, stim_combinations)

            new_event_id = modality_lexicality_event_ids[stim_combination]
            trial[:,-1] = new_event_id
            new_events_master.append(trial)
        except:
            print("Error with stim-code: {0}".format(stim_codes))
        
    return np.concatenate([x for x in new_events_master])

In [37]:
if 'Nasium' in epoch.ch_names:
    epoch.drop_channels(ch_names=['Nasium', 'LL4', 'L12', 'VEOG']);
copy_epoch = epoch.copy()
X = copy_epoch.pick_types(meg=False, eeg=True)
new_events = convert_epoch_events_to_stim_combinations(epoch_object= copy_epoch)    
copy_epoch.events = new_events

Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [14 18 13]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with stim-code: [20 21 19]
Error with

ValueError: need at least one array to concatenate

In [42]:
if 'Nasium' in epoch.ch_names:
    epoch.drop_channels(ch_names=['Nasium', 'LL4', 'L12', 'VEOG']);

In [45]:
X = epoch.get_data()
y = epoch.events[:,-1]
vec=Vectorizer()
X=vec.fit_transform(X)
print(X.shape,epoch.get_data().shape)

(1759, 32125) (1759, 125, 257)


In [46]:
from sklearn import metrics
clf = RandomForestRegressor(n_estimators =10, random_state = 42)
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 42)
    
clf.fit(X_train, y_train)
    
y_pred=clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred.round()))
score = clf.score(X_test, y_test)
print(score)

Accuracy: 1.0
1.0


In [47]:
y_pred

array([13.,  6., 23.,  4.,  7.,  4.,  5.,  9., 23.,  8., 21., 20.,  3.,
       11., 18., 11.,  5., 18., 22., 14., 11., 11.,  4., 11., 11., 24.,
        8., 22.,  3.,  4., 16.,  9., 16., 13., 19., 18., 13., 12., 12.,
        2.,  6., 19.,  4., 10., 17.,  6.,  5., 14., 22.,  3., 10., 24.,
       20., 12., 24., 11.,  8.,  6., 17., 10., 13.,  5., 20., 10., 10.,
       15.,  3., 10., 14., 11., 19.,  9., 21.,  7., 21., 21.,  5., 20.,
       18.,  9.,  6., 23.,  9.,  7., 11., 12.,  8., 19., 22., 18., 10.,
        6., 17., 13., 21., 11., 19., 19., 19.,  4.,  4., 23., 21.,  7.,
       20., 24., 15., 21., 21., 19.,  2.,  2.,  9.,  3., 12., 24., 19.,
       21., 19.,  3., 21., 19.,  8., 18., 22., 14.,  4.,  7.,  8., 13.,
       24., 21., 24.,  9., 16.,  2., 20.,  9., 13., 21., 20.,  4., 12.,
       18.,  4.,  9.,  5., 11., 19., 19., 14., 21., 23., 22., 10., 21.,
       18., 18., 11., 14., 22., 11.,  4.,  3., 22., 20.,  5., 13., 23.,
       12.,  5., 24.,  4.,  6.,  9., 13., 12., 22., 21.,  4., 13

In [48]:
epoch=filter_event_id(epoch)
print(epoch.events[:,-1])

[1 1 1 ... 1 1 1]


In [49]:
from sklearn import metrics
clf = RandomForestRegressor(n_estimators =10, random_state = 42)
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 42)
    
clf.fit(X_train, y_train)
    
y_pred=clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred.round()))
score = clf.score(X_test, y_test)
print(score)

Accuracy: 0.9159090909090909
0.6667614586818333


In [50]:
y_pred

array([0.9, 0.3, 0.1, 1. , 0.1, 1. , 1. , 0.2, 0.3, 0. , 0. , 0. , 1. ,
       0. , 1. , 0.1, 1. , 1. , 0.2, 1. , 0. , 0.3, 1. , 0. , 0. , 0. ,
       0.1, 0. , 1. , 1. , 0.9, 0.2, 1. , 0.7, 0. , 0.8, 0.7, 0.1, 0.1,
       1. , 0.4, 0. , 1. , 0. , 0.9, 0.3, 1. , 0.6, 0. , 1. , 0.2, 0.2,
       0.1, 0.1, 0. , 0.1, 0.1, 0.6, 1. , 0. , 0.7, 1. , 0. , 0. , 0.2,
       0.9, 1. , 0. , 0.9, 0.4, 0. , 0.1, 0.1, 0. , 0. , 0.1, 1. , 0. ,
       0.8, 0.4, 0.3, 0.2, 0.3, 0.1, 0.3, 0.4, 1. , 0.1, 0. , 0.9, 0. ,
       0.2, 1. , 0.8, 0.1, 0. , 0.2, 0.4, 0. , 1. , 1. , 0.2, 0. , 0.2,
       0. , 0.2, 1. , 0.2, 0.3, 0. , 1. , 1. , 0.2, 1. , 0.1, 0.1, 0.1,
       0.1, 0. , 1. , 0.2, 0.1, 0. , 0.7, 0.3, 1. , 1. , 0. , 0.3, 1. ,
       0.1, 0.3, 0.1, 0.5, 0.9, 0.9, 0.4, 0. , 1. , 0. , 0.1, 1. , 0.1,
       1. , 1. , 0. , 1. , 0.3, 0.2, 0. , 0.8, 0. , 0.2, 0.1, 0.1, 0.2,
       1. , 1. , 0. , 0.8, 0.1, 0.1, 1. , 1. , 0. , 0. , 1. , 0.9, 0.1,
       0. , 1. , 0. , 1. , 0.5, 0. , 1. , 0. , 0.1, 0.1, 1. , 1.

In [52]:
y_test

array([1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0,
       1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
       1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1,
       0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1,

In [53]:
import eli5

In [60]:
eli5.explain_weights(clf)

Weight,Feature
0.6669 ± 0.0664,x31868
0.0073 ± 0.0336,x22209
0.0056 ± 0.0173,x17645
0.0047 ± 0.0152,x18628
0.0042 ± 0.0191,x22123
0.0042 ± 0.0170,x31452
0.0042 ± 0.0169,x4015
0.0036 ± 0.0216,x23115
0.0034 ± 0.0138,x25583
0.0034 ± 0.0201,x3275


In [71]:
X=vec.fit_transform(X)
X[2].shape

(32125,)

In [89]:
eli5.show_prediction(clf, X[0])

In [75]:
from treeinterpreter import treeinterpreter as ti

In [77]:
prediction, bias, contributions = ti.predict(clf, X_test)

In [80]:
assert(numpy.allclose(prediction, bias + np.sum(contributions, axis=1)))
assert(numpy.allclose(clf.predict(X_test), bias + np.sum(contributions, axis=1)))

In [85]:
channel=[]
importances = np.array(clf.feature_importances_)
for i in range(0,len(importances)):
    if importances[i]>0.000001:
        channel.append(importances[i])
channel1=np.array(channel)
channel1.shape
importances.shape

(32125,)

In [95]:
X = epoch.get_data()
y = epoch.events[:,-1]
vec=Vectorizer()
X=vec.fit_transform(X)
print(X.shape,epoch.get_data().shape)

(1759, 32125) (1759, 125, 257)


In [96]:
clf = LogisticRegression()
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state = 42)
    
clf.fit(X_train, y_train)
    
score = clf.score(X_test, y_test)
print(score)

C:\Users\kingsly\Anaconda3\envs\mne\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


0.6041666666666666


In [91]:
print(score)

0.5587121212121212


In [97]:
eli5.show_prediction(clf, X[0])

In [98]:
eli5.sklearn.explain_weights.explain_linear_classifier_weights(clf)

Explanation(estimator="LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,\n          intercept_scaling=1, max_iter=100, multi_class='warn',\n          n_jobs=None, penalty='l2', random_state=None, solver='warn',\n          tol=0.0001, verbose=0, warm_start=False)", description="\nFeatures with largest coefficients.\nCaveats:\n1. Be careful with features which are not\n   independent - weights don't show their importance.\n2. If scale of input features is different then scale of coefficients\n   will also be different, making direct comparison between coefficient values\n   incorrect.\n3. Depending on regularization, rare features sometimes may have high\n   coefficients; this doesn't mean they contribute much to the\n   classification result for most examples.\n", error=None, method='linear model', is_regression=False, targets=[TargetExplanation(target=1, feature_weights=FeatureWeights(pos=[FeatureWeight(feature='<BIAS>', weight=1.5578843222786347, std=None, value=None), FeatureWeight(feature='x32115', weight=0.3056063271442402, std=None, value=None), FeatureWeight(feature='x31872', weight=0.06571537061734682, std=None, value=None), FeatureWeight(feature='x32078', weight=0.06360472219502097, std=None, value=None), FeatureWeight(feature='x32063', weight=0.0636047175297217, std=None, value=None), FeatureWeight(feature='x32087', weight=0.06360471462397824, std=None, value=None), FeatureWeight(feature='x32098', weight=0.06360471313674732, std=None, value=None), FeatureWeight(feature='x32071', weight=0.06322490088231916, std=None, value=None), FeatureWeight(feature='x32104', weight=0.06322489881793711, std=None, value=None)], neg=[FeatureWeight(feature='x32110', weight=-0.38189749450230315, std=None, value=None), FeatureWeight(feature='x32109', weight=-0.30652227035688795, std=None, value=None), FeatureWeight(feature='x31868', weight=-0.14142636685479895, std=None, value=None), FeatureWeight(feature='x32035', weight=-0.06608340516847744, std=None, value=None), FeatureWeight(feature='x32070', weight=-0.0649246168236568, std=None, value=None), FeatureWeight(feature='x32025', weight=-0.06430432605276755, std=None, value=None), FeatureWeight(feature='x32077', weight=-0.06410902379077939, std=None, value=None), FeatureWeight(feature='x32103', weight=-0.06381228843776869, std=None, value=None), FeatureWeight(feature='x31885', weight=-0.06267764669277917, std=None, value=None), FeatureWeight(feature='x31979', weight=-0.06267764491501547, std=None, value=None), FeatureWeight(feature='x32021', weight=-0.06267764452564208, std=None, value=None)], pos_remaining=16189, neg_remaining=15666), proba=None, score=None, weighted_spans=None)], feature_importances=None, decision_tree=None, highlight_spaces=None, transition_features=None)

In [99]:
eli5.explain_weights(clf)

Explanation(estimator="LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,\n          intercept_scaling=1, max_iter=100, multi_class='warn',\n          n_jobs=None, penalty='l2', random_state=None, solver='warn',\n          tol=0.0001, verbose=0, warm_start=False)", description="\nFeatures with largest coefficients.\nCaveats:\n1. Be careful with features which are not\n   independent - weights don't show their importance.\n2. If scale of input features is different then scale of coefficients\n   will also be different, making direct comparison between coefficient values\n   incorrect.\n3. Depending on regularization, rare features sometimes may have high\n   coefficients; this doesn't mean they contribute much to the\n   classification result for most examples.\n", error=None, method='linear model', is_regression=False, targets=[TargetExplanation(target=1, feature_weights=FeatureWeights(pos=[FeatureWeight(feature='<BIAS>', weight=1.5578843222786347, std=None, value=None), FeatureWeight(feature='x32115', weight=0.3056063271442402, std=None, value=None), FeatureWeight(feature='x31872', weight=0.06571537061734682, std=None, value=None), FeatureWeight(feature='x32078', weight=0.06360472219502097, std=None, value=None), FeatureWeight(feature='x32063', weight=0.0636047175297217, std=None, value=None), FeatureWeight(feature='x32087', weight=0.06360471462397824, std=None, value=None), FeatureWeight(feature='x32098', weight=0.06360471313674732, std=None, value=None), FeatureWeight(feature='x32071', weight=0.06322490088231916, std=None, value=None), FeatureWeight(feature='x32104', weight=0.06322489881793711, std=None, value=None)], neg=[FeatureWeight(feature='x32110', weight=-0.38189749450230315, std=None, value=None), FeatureWeight(feature='x32109', weight=-0.30652227035688795, std=None, value=None), FeatureWeight(feature='x31868', weight=-0.14142636685479895, std=None, value=None), FeatureWeight(feature='x32035', weight=-0.06608340516847744, std=None, value=None), FeatureWeight(feature='x32070', weight=-0.0649246168236568, std=None, value=None), FeatureWeight(feature='x32025', weight=-0.06430432605276755, std=None, value=None), FeatureWeight(feature='x32077', weight=-0.06410902379077939, std=None, value=None), FeatureWeight(feature='x32103', weight=-0.06381228843776869, std=None, value=None), FeatureWeight(feature='x31885', weight=-0.06267764669277917, std=None, value=None), FeatureWeight(feature='x31979', weight=-0.06267764491501547, std=None, value=None), FeatureWeight(feature='x32021', weight=-0.06267764452564208, std=None, value=None)], pos_remaining=16189, neg_remaining=15666), proba=None, score=None, weighted_spans=None)], feature_importances=None, decision_tree=None, highlight_spaces=None, transition_features=None)

In [100]:
epoch.drop_channels(['STI 014'])

<EpochsFIF  |   1759 events (all good), 0 - 0.5 sec, baseline [0, 0], ~428.0 MB, data loaded,
 'b-f1-pic': 0
 'b-f1-snd': 0
 'b-f1-spk': 0
 'b-f1-wrd': 0
 'b-f2-pic': 0
 'b-f2-snd': 0
 'b-f2-spk': 0
 'b-f2-wrd': 811
 'b-tg-pic': 0
 'b-tg-snd': 0
 'b-tg-spk': 0
 'b-tg-wrd': 0
 'f-f1-pic': 0
 'f-f1-snd': 0
 'f-f1-spk': 0
 'f-f1-wrd': 0
 'f-f2-pic': 0
 'f-f2-snd': 0
 'f-f2-spk': 0
 'f-f2-wrd': 0
 'f-tg-pic': 0
 'f-tg-snd': 0
 'f-tg-spk': 0
 'f-tg-wrd': 0>

In [101]:
X = epoch.get_data()
y = epoch.events[:,-1]
vec=Vectorizer()
X=vec.fit_transform(X)
print(X.shape,epoch.get_data().shape)

(1759, 31868) (1759, 124, 257)


In [102]:
clf = LogisticRegression()
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state = 42)
    
clf.fit(X_train, y_train)
    
score = clf.score(X_test, y_test)
print(score)

C:\Users\kingsly\Anaconda3\envs\mne\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


0.5549242424242424


In [103]:
eli5.explain_weights(clf)

Weight?,Feature
+0.000,x5717
+0.000,x970
+0.000,x5718
+0.000,x8113
+0.000,x945
+0.000,x5806
+0.000,x8112
+0.000,x8114
+0.000,x11583
+0.000,x5807


In [104]:
eli5.sklearn.explain_weights.explain_linear_classifier_weights(clf)

Weight?,Feature
+0.000,x5717
+0.000,x970
+0.000,x5718
+0.000,x8113
+0.000,x945
+0.000,x5806
+0.000,x8112
+0.000,x8114
+0.000,x11583
+0.000,x5807


In [105]:
from sklearn import metrics
clf = RandomForestRegressor(n_estimators =10, random_state = 42)
    
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 42)
    
clf.fit(X_train, y_train)
    
y_pred=clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred.round()))
score = clf.score(X_test, y_test)
print(score)

Accuracy: 0.5909090909090909
-0.028972900635664267


In [106]:
eli5.explain_weights(clf)

Weight,Feature
0.0087 ± 0.0354,x18054
0.0071 ± 0.0284,x28373
0.0068 ± 0.0270,x1450
0.0053 ± 0.0226,x29741
0.0049 ± 0.0197,x18071
0.0048 ± 0.0286,x30871
0.0047 ± 0.0281,x18819
0.0041 ± 0.0245,x17324
0.0040 ± 0.0243,x28413
0.0040 ± 0.0241,x17874


In [117]:
eli5.show_prediction(clf, X[7])

AttributeError: 'HTML' object has no attribute 'shape'